In [23]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

In [24]:
df = pd.read_csv("../data_set/hiring.csv")
df

,experience,test_score(out of 10),interview_score(out of 10),salary($)
0,NaN,8.0,9,50000
1,NaN,8.0,6,45000
2,five,6.0,7,60000
3,two,10.0,10,65000
4,seven,9.0,6,70000
5,three,7.0,10,62000
6,ten,NaN,7,72000
7,eleven,7.0,8,80000


In [25]:
# Rename columns
df = df.rename(
    columns={
        "experience": "Experience",
        "test_score(out of 10)": "Test_Score",
        "interview_score(out of 10)": "Interview_Score",
        "salary($)": "Salary",
    }
)
df

,Experience,Test_Score,Interview_Score,Salary
0,NaN,8.0,9,50000
1,NaN,8.0,6,45000
2,five,6.0,7,60000
3,two,10.0,10,65000
4,seven,9.0,6,70000
5,three,7.0,10,62000
6,ten,NaN,7,72000
7,eleven,7.0,8,80000


In [26]:
ct = ColumnTransformer(
    transformers = [
        ("exprience", SimpleImputer(strategy="constant",fill_value='zero'), ["Experience",]),
        ("salary", SimpleImputer(strategy="median"), ["Test_Score",],),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False, # Stops scikit-learn from prefixing 'step__' to columns
).set_output(transform='pandas')

df = ct.fit_transform(df)
df

,Experience,Test_Score,Interview_Score,Salary
0,zero,8.0,9,50000
1,zero,8.0,6,45000
2,five,6.0,7,60000
3,two,10.0,10,65000
4,seven,9.0,6,70000
5,three,7.0,10,62000
6,ten,8.0,7,72000
7,eleven,7.0,8,80000


In [27]:
X = df.drop(columns=["Salary"])
X

,Experience,Test_Score,Interview_Score
0,zero,8.0,9
1,zero,8.0,6
2,five,6.0,7
3,two,10.0,10
4,seven,9.0,6
5,three,7.0,10
6,ten,8.0,7
7,eleven,7.0,8


In [28]:
y = df["Salary"]
print(y)

0    50000
1    45000
2    60000
3    65000
4    70000
5    62000
6    72000
7    80000
Name: Salary, dtype: int64


In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [30]:
X_train

,Experience,Test_Score,Interview_Score
7,eleven,7.0,8
2,five,6.0,7
4,seven,9.0,6
3,two,10.0,10
6,ten,8.0,7


In [31]:
X_test

,Experience,Test_Score,Interview_Score
1,zero,8.0,6
5,three,7.0,10
0,zero,8.0,9


In [32]:
y_train

7    80000
2    60000
4    70000
3    65000
6    72000
Name: Salary, dtype: int64

In [33]:
y_test

1    45000
5    62000
0    50000
Name: Salary, dtype: int64

In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num_col", StandardScaler(), ["Test_Score", "Interview_Score",]),
        ("cat_col", OneHotEncoder(sparse_output=False,handle_unknown='ignore'), ["Experience",]),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
).set_output(transform='pandas')

In [35]:
pipeline = Pipeline(
    steps=[
    ('preprocessing',preprocessor),
    ('model',LinearRegression())
]
)

In [36]:
pipeline.fit(X_train, y_train)

model = pipeline.named_steps['model']

print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

Intercept: 69400.0
Coefficients: [  155.40808378  -447.18659229 10841.75824176 -9378.02197802
   -37.36263736  2402.1978022  -3828.57142857]


In [37]:
predictions = pipeline.predict(X_test)

print(predictions)

[69927.47252747 68498.9010989  68938.46153846]


In [38]:
frature_name = X_train.columns
frature_name

Index(['Experience', 'Test_Score', 'Interview_Score'], dtype='str')

In [39]:

new_data = pd.DataFrame([["eleven",7,8]],columns=frature_name)

salary = pipeline.predict(new_data)

print(salary)

[80000.]


In [40]:
# 1. Transform the input using ONLY the preprocessing step
transformed_data = pipeline.named_steps["preprocessing"].transform(new_data)

# 2. View the internal array with column headers
print(transformed_data)

   Test_Score  Interview_Score  Experience_eleven  Experience_five  \
0   -0.707107         0.294884                1.0              0.0   

   Experience_seven  Experience_ten  Experience_two  
0               0.0             0.0             0.0  


In [41]:
# Evaluate performance on the Training Set (Should be high because the model practiced on this)
train_accuracy = pipeline.score(X_train, y_train)

# Evaluate performance on the Testing Set (The true test of how well the model generalizes to new data)
test_accuracy = pipeline.score(X_test, y_test)

print(f"Training Accuracy: {train_accuracy * 100:.1f}%")
print(f"Testing Accuracy:  {test_accuracy * 100:.1f}%")

Training Accuracy: 100.0%
Testing Accuracy:  -569.6%
